[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/26_recommender_systems/01_popularity_and_content_based.ipynb)

# Bài 01 — Bài toán gợi ý, popularity và content-based

**🎯 Mục tiêu**
- Các dạng bài toán gợi ý: phản hồi **tường minh** (rating) vs **ngầm** (click, xem); **dự đoán rating** vs **xếp hạng top-N**.
- Ma trận user–item thưa, phân phối **đuôi dài**; giao thức đánh giá (holdout theo từng user, precision@k, recall@k).
- **Popularity baseline**: đếm, trung bình, trung bình **có giảm chấn** (Bayesian average).
- **Content-based**: hồ sơ item → hồ sơ user → độ tương đồng; ưu/nhược; lai với popularity.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/26_recommender_systems

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ml_course.metrics import ndcg_at_k, precision_at_k, recall_at_k
from ml_course.visualization import set_style

set_style()

## 1. Dữ liệu: rating phim

`ratings.csv`: (user, phim, rating 1–5, thời điểm). `items.csv`: tên, thể loại chính, năm.

In [ ]:
ratings = pd.read_csv("../../data/sample/ratings.csv")
items = pd.read_csv("../../data/sample/items.csv")
n_users, n_items = ratings.user_id.max() + 1, ratings.item_id.max() + 1
density = len(ratings) / (n_users * n_items)
print(f"{len(ratings):,} rating | {n_users} user × {n_items} phim | mật độ ma trận = {density:.1%}")
print("phân phối rating:", ratings.rating.value_counts().sort_index().to_dict())
print(items.head(3).to_string(index=False))

In [ ]:
item_counts = ratings.item_id.value_counts()
user_counts = ratings.user_id.value_counts()
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(range(len(item_counts)), item_counts.values, width=1.0)
top20_share = item_counts.iloc[: int(0.2 * n_items)].sum() / len(ratings)
axes[0].set_title(f"Đuôi dài: 20% phim phổ biến nhất chiếm {top20_share:.0%} số rating")
axes[0].set_xlabel("phim (sắp theo số rating)")
axes[1].hist(user_counts.values, bins=30)
axes[1].set_title(f"Số rating mỗi user: trung vị {int(user_counts.median())}, nhiều nhất {user_counts.max()}")
plt.show()

Hai đặc điểm của dữ liệu gợi ý thật:
- **Thưa**: mỗi user chỉ rate vài phần trăm số item — 91% ô của ma trận user–item trống.
- **Đuôi dài**: một số ít item chiếm phần lớn tương tác; đa số item hiếm khi được thấy.

| | Phản hồi tường minh | Phản hồi ngầm |
|---|---|---|
| Ví dụ | rating 1–5, like/dislike | click, xem, mua, thời gian xem |
| Lượng dữ liệu | ít | rất nhiều |
| Tín hiệu tiêu cực | có | không rõ (không click ≠ không thích) |
| Bài toán thường gặp | dự đoán rating | xếp hạng top-N |

Sản phẩm thật gần như luôn cần **top-N**: "10 phim nên hiện cho user này" — không phải "user sẽ chấm phim X mấy điểm".

## 2. Giao thức đánh giá

Với mỗi user có ≥ 5 rating, giữ lại ngẫu nhiên 20% rating làm **test**. Một gợi ý được tính là **đúng** nếu là phim user đã xem trong test và chấm **≥ 4**. Ứng viên gợi ý: mọi phim user **chưa** thấy trong train.

In [ ]:
def split_per_user(ratings, test_frac=0.2, min_ratings=5, seed=0):
    rng = np.random.default_rng(seed)
    test_mask = np.zeros(len(ratings), bool)
    for _, g in ratings.groupby("user_id"):
        if len(g) >= min_ratings:
            test_mask[rng.choice(g.index, max(1, int(test_frac * len(g))), replace=False)] = True
    return ratings[~test_mask], ratings[test_mask]


train, test = split_per_user(ratings)
seen = {u: set(g.item_id) for u, g in train.groupby("user_id")}
test_ratings = {u: dict(zip(g.item_id, g.rating)) for u, g in test.groupby("user_id")}
print(f"train {len(train):,} | test {len(test):,} | user được đánh giá (có ≥ 1 phim thích trong test): "
      f"{sum(any(r >= 4 for r in d.values()) for d in test_ratings.values())}")


def evaluate_topk(score_fn, k=10):
    """score_fn(u) → điểm cho mọi item; loại phim đã thấy trong train; trả về precision/recall/NDCG@k trung bình."""
    prec, rec, ndcg = [], [], []
    for u, rated in test_ratings.items():
        liked = [i for i, r in rated.items() if r >= 4]
        if not liked:
            continue
        scores = np.asarray(score_fn(u), float).copy()
        scores[list(seen.get(u, []))] = -np.inf
        top = np.argsort(-scores)[:k]
        prec.append(precision_at_k(top, liked, k))
        rec.append(recall_at_k(top, liked, k))
        ndcg.append(ndcg_at_k(top, {i: r - 3 for i, r in rated.items() if r >= 4}, k))
    return {"precision@10": np.mean(prec), "recall@10": np.mean(rec), "NDCG@10": np.mean(ndcg)}

(Chi tiết các metric xếp hạng ở bài 04.) Mốc tham chiếu: gợi ý **ngẫu nhiên**.

In [ ]:
results = {"ngẫu nhiên": evaluate_topk(lambda u: np.random.default_rng(u).random(n_items))}
pd.DataFrame(results).T.round(4)

## 3. Popularity baseline

Gợi ý cho mọi user cùng một danh sách các phim "tốt nhất" — nhưng "tốt nhất" theo nghĩa nào?

- **Đếm**: nhiều rating nhất.
- **Trung bình rating**: phim có 1 rating 5 sao đứng đầu → nhiễu.
- **Trung bình giảm chấn** (Bayesian average): kéo trung bình của phim ít rating về trung bình chung $\mu$:
$$\tilde r_i = \frac{\sum_u r_{ui} + m\,\mu}{n_i + m}$$
$m$ = "số rating ảo" bằng $\mu$ — phim cần nhiều rating thật mới vượt khỏi $\mu$.

In [ ]:
cnt = train.groupby("item_id").size().reindex(range(n_items), fill_value=0).to_numpy()
total = train.groupby("item_id").rating.sum().reindex(range(n_items), fill_value=0).to_numpy()
mu = train.rating.mean()
raw_mean = np.where(cnt > 0, total / np.maximum(cnt, 1), 0)
damped = (total + 10 * mu) / (cnt + 10)
for name, s in [("popularity: số rating", cnt), ("popularity: trung bình thô", raw_mean), ("popularity: trung bình giảm chấn (m = 10)", damped)]:
    results[name] = evaluate_topk(lambda u, s=s: s)
    print(f"{name:42s} số rating của 5 phim đầu bảng: {cnt[np.argsort(-s)[:5]].tolist()}")
pd.DataFrame(results).T.round(4)

- Trung bình **thô** đưa các phim có 1–8 rating lên đầu → gần như ngẫu nhiên.
- Giảm chấn đỡ hơn, nhưng **đếm** thắng xa: trong dữ liệu này (như thực tế), phim phổ biến là phim **được xem** — và chỉ phim được xem mới có thể xuất hiện trong test. Popularity là baseline **khó đánh bại**, mọi mô hình cá nhân hoá phải vượt nó.
- Nhược điểm: không cá nhân hoá, mọi người thấy cùng danh sách, các phim đuôi dài không bao giờ được gợi ý (bài 04 đo bằng **độ phủ**).

## 4. Content-based

Gợi ý phim **giống** những phim user đã thích, dựa trên **đặc trưng nội dung** của item:
1. **Hồ sơ item** $\mathbf x_i$: one-hot thể loại (+ năm, mô tả TF-IDF, diễn viên... nếu có).
2. **Hồ sơ user** $\mathbf p_u = \sum_{i \in I_u} w_{ui}\,\mathbf x_i$ với $w_{ui}$ = rating đã trừ trung bình của user (thích hơn mức thường → trọng số dương).
3. Điểm $s_{ui} = \cos(\mathbf p_u, \mathbf x_i)$.

In [ ]:
genre_onehot = pd.get_dummies(items.set_index("item_id").genre).reindex(range(n_items)).to_numpy(float)
X_items = genre_onehot / np.linalg.norm(genre_onehot, axis=1, keepdims=True)
R_train = np.full((n_users, n_items), np.nan)
R_train[train.user_id, train.item_id] = train.rating


def user_profile(u):
    rated = list(seen.get(u, []))
    weights = R_train[u, rated] - np.nanmean(R_train[u]) + 0.5         # +0.5: rating trung bình của user vẫn hơi tích cực
    return weights @ X_items[rated]


def content_scores(u):
    p = user_profile(u)
    return X_items @ (p / (np.linalg.norm(p) + 1e-9))


u0 = 7
profile = pd.Series(user_profile(u0), index=sorted(items.genre.unique())).round(2)
print(f"hồ sơ thể loại của user {u0}:", profile.to_dict())
top = [i for i in np.argsort(-content_scores(u0)) if i not in seen[u0]][:5]
print("gợi ý:", items.loc[top, ["title", "genre"]].values.tolist())
results["content-based (thể loại)"] = evaluate_topk(content_scores)
pd.DataFrame(results).T.round(4)

Content-based tốt hơn ngẫu nhiên nhưng kém popularity: thể loại là đặc trưng **quá thô** — trong một thể loại có hàng chục phim, mô hình không biết chọn phim nào, và nó bỏ qua tín hiệu mạnh nhất (phim nào được xem nhiều). Kết hợp hai nguồn — điểm nội dung chuẩn hoá cộng log độ phổ biến:

In [ ]:
def hybrid_scores(u, weight=1.0):
    s = content_scores(u)
    return np.log1p(cnt) + weight * (s - s.mean()) / (s.std() + 1e-9)


results["lai: popularity + content"] = evaluate_topk(hybrid_scores)
pd.DataFrame(results).T.round(4)

Bản lai có precision và đặc biệt NDCG cao hơn popularity (phim đúng được xếp **cao hơn** trong danh sách), recall hơi thấp hơn — tức là ngang ngửa về số phim đúng tìm được nhưng **cá nhân hoá** được. Bài 02–03 dùng chính **hành vi** của user khác (collaborative filtering) thay vì đặc trưng nội dung.

| Content-based | |
|---|---|
| ✅ | gợi ý được **item mới** chưa ai tương tác (cold start item) — chỉ cần đặc trưng |
| ✅ | giải thích được ("vì bạn thích phim hài") |
| ✅ | không cần dữ liệu của user khác |
| ❌ | phụ thuộc chất lượng đặc trưng (cần NLP/CV cho mô tả, ảnh — chương 32–34) |
| ❌ | "bong bóng lọc": chỉ gợi ý cái giống cái đã thích, ít bất ngờ |
| ❌ | user mới chưa có lịch sử → không có hồ sơ (cold start user) |

## 🧠 Kiểm tra nhanh

1. Phản hồi tường minh và ngầm khác nhau thế nào? Bài toán top-N khác dự đoán rating ra sao?
2. Vì sao trung bình rating thô là cách xếp hạng tồi? Bayesian average sửa thế nào?
3. Vì sao popularity là baseline mạnh trong đánh giá offline?
4. Hồ sơ user trong content-based được tạo thế nào?
5. Content-based giải quyết loại cold start nào, không giải quyết loại nào?

## 📌 Tóm tắt

- Dữ liệu gợi ý thưa, đuôi dài; sản phẩm cần top-N; đánh giá bằng holdout từng user + metric xếp hạng.
- Popularity (đếm) là baseline khó vượt; trung bình phải giảm chấn.
- Content-based: hồ sơ item → hồ sơ user → cosine; giải thích được, xử lý item mới; hạn chế bởi đặc trưng; lai với popularity.

## 📚 Tài liệu tham khảo

- Aggarwal (2016) — *Recommender Systems: The Textbook*, ch. 1 & 4
- Ricci, Rokach & Shapira (eds.) — *Recommender Systems Handbook* (3rd ed., 2022)
- Lops, de Gemmis & Semeraro (2011) — *Content-based Recommender Systems: State of the Art and Trends*